But: à partir d'un bilan pdf donné par l'utilisateur, le système fait une analyse de l'entreprise afin d'identifier les potentiels risques lié au révisorat d'entreprise

Prérequis: disposer d'une clé OpenAI

Fonctionnement:
- Modifier le pdf path pour mettre le path du bilan pdf 
- Compléter le fichier de configuration config.yaml avec la bonne clé et le model choisi
- Executer les différentes cellules pour tester l'analyse faite sur ce document selon chaque prompt.

# Import main packages

In [ ]:
# Put the path of your pdf
pdf_path = # to complete 

In [ ]:
# Import main packages
from utils import read_yaml, define_llm, retrieve_document_object

from rich.markdown import Markdown

import nest_asyncio
nest_asyncio.apply()

In [ ]:
# import config
config = read_yaml("../config.yaml")

# Get documents

### 1) Pour un seul document pdf

In [ ]:
# Pour un seul document pdf (si un ensemble de documents, tout mettre dans le même répertoire et remplacer pdf_path par le path du répertoire)
print("Loading...")
doc_pdf = retrieve_document_object(pdf_path)
len(doc_pdf)

## Filter to keep only the relevant pages

In [ ]:
def extract_relevant_pages(doc_pdf):
  """
  Extracts pages from a list of text strings, starting from "bilan" and ending at "compte de résultat".

  Args:
      doc_pdf: A list of strings, where each string represents a page of the PDF.

  Returns:
      A list of strings representing the relevant pages.  Returns an empty list if no relevant pages are found.
  """
  start_index = -1
  end_index = -1
  relevant_pages = []

  for i, page in enumerate(doc_pdf):
    if (("COMPTES ANNUELS" in page.text and "BILAN APRÈS RÉPARTITION" in page.text) or ("JAARREKENING" in page.text and "BALANS NA WINSTVERDELING" in page.text)) and start_index == -1:

      start_index = i

    if ("ANNEXE" in page.text) or ("TOELICHTING" in page.text and "XXXX" in page.text):
      end_index = i - 1
      break

  if start_index != -1:
    if end_index != -1 and end_index > start_index:
      relevant_pages = doc_pdf[start_index:end_index +1]
    else:
      relevant_pages = doc_pdf[start_index:]

  return relevant_pages

In [ ]:
# ici on garde juste les pages qui nous interesse pour le doc donné
relevant_docs = extract_relevant_pages(doc_pdf)

# Now relevant_docs contains only the pages between "bilan" and "compte de résultat"
# If no pages are found then relevant_docs will be empty
if relevant_docs:
  print(
      f"We get {len(relevant_docs)} relevant pages : pages {relevant_docs[0].metadata['page_label']} to {relevant_docs[-1].metadata['page_label']}")

else:
  print("No relevant pages were found in the document.")

# Define model
## with OpenAI

In [ ]:
openai_key = config["openai_key"]

llm = define_llm(config["llm_chosen"], secret_key=openai_key)

# Define prompt

## Simple context

In [ ]:
prompt = """
#INSTRUCTIONS
Tu es un auditeur d'entreprise et je te donne ci-dessous une balance sheet. Serts toi de ce document pour analyser les risques de l'entreprise et détaille tes calcules avec les formules et les réponses. **Ta réponse doit être en français**
#BALANCE_SHEET
{docs}
"""

## Simple context with more instructions

In [ ]:
prompt2 = """
# INSTRUCTIONS
Vous êtes un auditeur d'entreprise et je te donne ci-dessous une balance sheet. En utilisant ce document, calculez et analyse les ratios financiers clés, tels que les ratios de liquidité, de rentabilité et de levier, pour une vision globale de la santé financière. Détaille bien tes calculs.**Ta réponse doit être en français**
# BALANCE_SHEET
{docs}
"""

## Context with example

In [ ]:
Example = """
## Input
Supposons une entreprise avec les données suivantes sur l’exercice actuel :
- Chiffre d’affaires : 1 000 000 €
- Résultat net : 50 000 €
- Fonds propres : 300 000 €
- Dettes financières : 200 000 €
- Actif total : 600 000 €
- Passif courant : 150 000 €
- Actif courant : 200 000 €
- Stocks : 50 000 €

## Output
Voici une analyse détaillée des principaux ratios financiers pour cette entreprise :

---

### 1. **Analyse de la rentabilité**
\[
\text{Marge nette} = \left( \frac{\text{Résultat net}}{\text{Chiffre d'affaires}} \right) \times 100 = \left( \frac{50\,000}{1\,000\,000} \right) \times 100 = 5\%
\]
- Une marge nette de 5 % indique une rentabilité modeste mais acceptable selon les standards de nombreuses industries.

---

### 2. **Analyse de la solvabilité**
\[
\text{Ratio de solvabilité} = \frac{\text{Fonds propres}}{\text{Actif total}} = \frac{300\,000}{600\,000} = 0.5 \, (ou \, 50\%)
\]
- Avec 50 % de fonds propres, l’entreprise est correctement capitalisée. Cela montre une certaine stabilité financière.

\[
\text{Ratio d’endettement} = \frac{\text{Dettes financières}}{\text{Fonds propres}} = \frac{200\,000}{300\,000} = 0.67
\]
- Ce ratio est inférieur à 1, ce qui signifie que l’entreprise finance une partie significative de ses actifs via ses fonds propres, ce qui est généralement positif.

---

### 3. **Analyse de la liquidité**
\[
\text{Ratio de liquidité générale} = \frac{\text{Actif courant}}{\text{Passif courant}} = \frac{200\,000}{150\,000} = 1.33
\]
- Un ratio supérieur à 1 signifie que l’entreprise peut couvrir ses dettes à court terme, mais la marge est étroite.

\[
\text{Ratio de liquidité immédiate} = \frac{\text{Actif courant} - \text{Stocks}}{\text{Passif courant}} = \frac{200\,000 - 50\,000}{150\,000} = \frac{150\,000}{150\,000} = 1
\]
- Avec un ratio de 1, l’entreprise peut rembourser ses dettes à court terme sans dépendre de la vente de stocks. C’est rassurant.

---

### 4. **Analyse de l’activité**
Donnée manquante : coût des ventes. Si nous supposons un coût des ventes estimé de 700 000 €, la rotation serait :
\[
\text{Rotation des stocks} = \frac{\text{Coût des ventes}}{\text{Stocks}} = \frac{700\,000}{50\,000} = 14
\]
- Un ratio de 14 indique que les stocks tournent rapidement, ce qui est souvent signe d’une bonne gestion.

---

### Résumé des résultats :
| **Indicateur**                  | **Valeur**   | **Interprétation**                                                                                                                                               |
|----------------------------------|--------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Marge nette                      | 5 %          | Rentabilité correcte.                                                                                                                                             |
| Ratio de solvabilité             | 50 %         | Bonne structure financière, équilibrée entre dettes et fonds propres.                                                                                             |
| Ratio d’endettement              | 0.67         | Endettement raisonnable, maîtrisé.                                                                                                                                |
| Ratio de liquidité générale      | 1.33         | Liquidité suffisante pour couvrir les dettes à court terme.                                                                                                       |
| Ratio de liquidité immédiate     | 1            | Bonne capacité à rembourser les dettes à court terme sans avoir à vendre de stocks.                                                                                |
| Rotation des stocks (estimée)    | 14 fois      | Gestion rapide et efficace des stocks.                                                                                                                            |

---

### **Conclusion**
L’entreprise est globalement bien gérée avec une bonne rentabilité et une structure financière solide. Les principaux risques pourraient provenir d’un resserrement de la liquidité en cas de chocs externes ou d’une pression concurrentielle réduisant la marge nette. Une surveillance du passif courant et de l’évolution des stocks est recommandée.
"""

In [ ]:
prompt3 = """
# INSTRUCTIONS
You are a company auditor, and I am providing you with a balance sheet that you use to evaluate the company's risks by detailing your calculations. I would like your answer to include the following six parts. **Ta réponse doit être en français**:
- Analyse de la rentabilité
- Analyse de la solvabilité
- Analyse de la liquidité
- Analyse des performances opérationnelles
- Risques identifiés
- Conclusion

# Example
Below is a fictitious example of a balance sheet analysis.
{Example}
# BALANCE_SHEET
{docs}"""

# Query

## En utilisant l'ensemble du document

In [ ]:
# Define the prompt
prompt_complete_question = prompt.format(
  docs=doc_pdf
  )

# The model creates a response
response = llm.complete(prompt_complete_question)

Markdown(response.text)

In [ ]:
# Define the prompt
prompt_complete_question = prompt2.format(
  docs=doc_pdf
  )

# The model creates a response
response = llm.complete(prompt_complete_question)

Markdown(response.text)

In [ ]:
# Define the prompt
prompt_complete_question = prompt3.format(
  docs=doc_pdf,
  Example=Example
  )

# The model creates a response
response = llm.complete(prompt_complete_question)

Markdown(response.text)

## En utilisant seulement les parties relevants (Bilan et compte de résultats)

In [ ]:
# Define the prompt
prompt_complete_question = prompt.format(
  docs=relevant_docs
  )

# The model creates a response
response = llm.complete(prompt_complete_question)

Markdown(response.text)

In [ ]:
# Define the prompt
prompt_complete_question = prompt2.format(
  docs=relevant_docs
  )


# The model creates a response
response = llm.complete(prompt_complete_question)

Markdown(response.text)

In [ ]:
# Define the prompt
prompt_complete_question = prompt3.format(
  docs=relevant_docs,
  Example=Example
  )


# The model creates a response
response = llm.complete(prompt_complete_question)

Markdown(response.text)